In [49]:
import pandas as pd
import json
import pickle
import duckdb
import unicodedata
import requests

In [50]:
API_URL = "https://ccs-lab.zcu.cz/grela-api/api/query"
DUCKDB_PATH = "/srv/data/grela/grela_v0.6.duckdb"

def run_query_df(query: str, mode: str = "api") -> pd.DataFrame:
    if mode == "api":
        resp = requests.post(API_URL, json={"query": query})

        if resp.status_code != 200:
            # Helpful debug if something goes wrong
            try:
                print("API error JSON:", resp.json())
            except Exception:
                print("API non-JSON error:", resp.text[:1000])
            resp.raise_for_status()

        data = resp.json()

        # If the server explicitly says it failed
        if not data.get("success", False) and "error" in data:
            raise RuntimeError(f"Server error: {data}")

        # Empty result case: success=True but no download_url
        if "download_url" not in data:
            # You can preserve columns if you want:
            cols = [c["name"] for c in data.get("columns", [])]
            return pd.DataFrame(columns=cols)

        # Normal case: we have a parquet file to read
        return pd.read_parquet(data["download_url"])

    else:
        conn = duckdb.connect(DUCKDB_PATH, read_only=True)
        df = conn.execute(query).fetchdf()
        conn.close()
        return df

In [51]:
def get_enemy_works(mode: str = "api") -> pd.DataFrame:
    query = r"""
    SELECT
        grela_id,
        author,
        title,
        not_before,
        not_after,
        provenience,
        CASE
            WHEN provenience = 'christian'
             AND not_before >= 0   AND not_before < 300
             AND not_after  >= 0   AND not_after  < 300
            THEN 'christian_0_300'

            WHEN provenience = 'christian'
             AND not_before >= 300 AND not_before < 600
             AND not_after  >= 300 AND not_after  < 600
            THEN 'christian_300_600'

            WHEN provenience = 'pagan'
             AND not_before >= 0   AND not_before < 300
             AND not_after  >= 0   AND not_after  < 300
            THEN 'pagan_0_300'

            WHEN provenience = 'pagan'
             AND not_before >= 300 AND not_before < 600
             AND not_after  >= 300 AND not_after  < 600
            THEN 'pagan_300_600'
        END AS enemy_subcorpus
    FROM works
    WHERE provenience IN ('christian', 'pagan')
    """
    df = run_query_df(query, mode)
    # keep only rows that actually fall into one of the four bins
    return df[df["enemy_subcorpus"].notna()]

In [52]:
works_df = get_enemy_works("api")
print(works_df["enemy_subcorpus"].value_counts())

enemy_subcorpus
pagan_0_300          602
pagan_300_600        157
christian_0_300      126
christian_300_600     59
Name: count, dtype: int64


In [53]:
def get_sentence_groups_for_work(grela_id: str, mode: str = "api") -> pd.DataFrame:
    # basic escaping for safety
    grela_sql = str(grela_id).replace("'", "''")

    query = f"""
    SELECT
        t.grela_id,
        t.sentence_id,
        s.position   AS sentence_position,
        s.sent_text,
        LIST(
            STRUCT_PACK(
                token_id    := t.token_id,
                token_text  := t.token_text,
                lemma       := t.lemma,
                pos         := t.pos,
                ref         := t.ref,
                char_start  := t.char_start,
                char_end    := t.char_end
            )
            ORDER BY t.token_id
        ) AS tokens
    FROM tokens t
    LEFT JOIN sentences s USING (grela_id, sentence_id)
    WHERE t.grela_id = '{grela_sql}'
    GROUP BY
        t.grela_id,
        t.sentence_id,
        s.position,
        s.sent_text
    """

    return run_query_df(query, mode)

In [54]:
def get_grouped_enemy_sentences(mode: str = "api") -> pd.DataFrame:
    works = get_enemy_works(mode)
    frames = []

    for row in works.itertuples(index=False):
        grela_id = row.grela_id
        label    = row.enemy_subcorpus

        df = get_sentence_groups_for_work(grela_id, mode)

        if df.empty:
            continue

        # Attach work-level metadata and subcorpus to each sentence row
        df["author"]          = row.author
        df["title"]           = row.title
        df["not_before"]      = row.not_before
        df["not_after"]       = row.not_after
        df["provenience"]     = row.provenience
        df["enemy_subcorpus"] = label

        frames.append(df)

    if not frames:
        return pd.DataFrame()

    grouped = pd.concat(frames, ignore_index=True)
    return grouped

In [55]:
grouped = get_grouped_enemy_sentences("api")
grouped["enemy_subcorpus"].value_counts()

enemy_subcorpus
pagan_0_300          545262
christian_300_600    176213
pagan_300_600        157188
christian_0_300      122638
Name: count, dtype: int64

In [59]:
# normalize token and lemma text within tokens column...
# If you want to keep diaeresis, include "\u0308".
KEEP_COMBINING = set()  # e.g., set(["\u0308"]) to keep diaeresis

def normalize_greek(text: str, *, strip_diacritics: bool = True) -> str:
    if not isinstance(text, str):
        return text
    # 1) Decompose so diacritics become combining marks
    s = unicodedata.normalize("NFD", text)

    # 2) Unify variant apostrophes/koronis to a single mark (’)
    #    This helps reduce visual variants in Greek texts
    s = (
        s.replace("\u1FBD", "’")  # Greek Koronis
        .replace("\u02BC", "’")  # Modifier Letter Apostrophe
        .replace("\u2019", "’")  # Right single quotation mark
        .replace("\u00B4", "’")  # Spacing acute accent (rarely used)
        .replace("'", "’")       # ASCII apostrophe
    )

    # 3) Optionally strip all combining marks (accents, breathings, subscripts)
    if strip_diacritics:
        s = "".join(
            ch for ch in s
            if not (unicodedata.category(ch) == "Mn" and ch not in KEEP_COMBINING)
        )

    # 4) Recompose
    return unicodedata.normalize("NFC", s)

def norm_tokens_lemmata(tokens, *, strip_diacritics: bool = True):
    out = []
    for t in tokens or []:
        if not isinstance(t, dict):
            continue
        d = dict(t)  # avoid mutating the original
        if "lemma" in d and isinstance(d["lemma"], str):
            d["lemma"] = normalize_greek(d["lemma"], strip_diacritics=strip_diacritics)
        if "token" in d and isinstance(d["token"], str):
            d["token"] = normalize_greek(d["token"], strip_diacritics=strip_diacritics)
        out.append(d)
    return out


# apply the normalization
# let's try what happens if we skip it....
#  grouped["tokens"] = grouped["tokens"].apply(lambda toks: norm_tokens_lemmata(toks, strip_diacritics=False))

In [ ]:
def get_lemma_sentence(tokens):
    try:
        lemma_sentence = [t["lemma"] for t in tokens if t["pos"] in ["NOUN", "VERB", "ADJ", "NOUN"]]
    except:
        lemma_sentence = []
    return lemma_sentence
grouped["lemma_sentence"] = grouped["tokens"].apply(get_lemma_sentence)

In [60]:
grouped.sample(10, random_state=0)

,grela_id,sentence_id,sentence_position,sent_text,tokens,author,title,not_before,not_after,provenience,enemy_subcorpus,lemma_sentence
936902,lagt_tlg4017.tlg001,lagt_tlg4017.tlg001_3845,3845,εἰ μὴ ἄρα κατὰ σὲ τὸν ποιητικὴν αἰτίαν ἐπὶ τῶν...,"[{'char_end': 2, 'char_start': 0, 'lemma': 'εἰ...",Syrianus,In Aristotelis Metaphysica Commentaria,401,450,pagan,pagan_300_600,"[ποιητικός, αἰτία, ἀίδιος, προσδέχομαι]"
624490,lagt_tlg2018.tlg001,lagt_tlg2018.tlg001_5500,5500,"εἰ γὰρ μη δὲ ἑαυτῶν, σχολῇ γ᾿ ἂν τῶν ἡμετέρων ...","[{'char_end': 2, 'char_start': 0, 'lemma': 'εἰ...",Eusebius,Praeperatio Evangelica,312,322,christian,christian_300_600,"[σχολή, ἡμέτερος, ἀπέχω]"
204167,lagt_tlg0057.tlg099,lagt_tlg0057.tlg099_1630,1630,τὸ δ ἐστὶ τοιοῦτο ὅτι τῶν ζητουμένων ἔνια μηδὲ...,"[{'char_end': 2, 'char_start': 0, 'lemma': 'ὁ'...",Galen,In Hippocratis Prognosticum,129,216,pagan,pagan_0_300,"[εἰμί, ζητέω, ἔνιοι, τέχνη, ἔργον, συμβάλλω, λ..."
450725,lagt_tlg0638.tlg001,lagt_tlg0638.tlg001_1494,1494,"ἐνταῦθα δείσας ὁ Δάμις περὶ τοῖς νέοις, μὴ χεί...","[{'char_end': 7, 'char_start': 0, 'lemma': 'ἐν...",Philostratus the Athenian,Vita Apollonii,170,215,pagan,pagan_0_300,"[δείδω, Δάμις, νέος, κακός, γίγνομαι, Φιλόλαος..."
297039,lagt_tlg0099.tlg001,lagt_tlg0099.tlg001_689,689,"αὐτὰρ ἐγὼ ποτὶ μακρὸν ἐρινεὸν ὑψόσ ἀερθείς, τῷ...","[{'char_end': 5, 'char_start': 0, 'lemma': 'ἀτ...",Strabo,Γεωγραφικά,18,24,pagan,pagan_0_300,"[μακρός, ἐρινεός, ἀείρω, προσφύω, ἔχω, νυκτερίς]"
887483,lagt_tlg4015.tlg007,lagt_tlg4015.tlg007_3873,3873,"καί φησιν, εἴ τι τῶν ζῴων ἔχει μὲν προβολὴν το...","[{'char_end': 3, 'char_start': 0, 'lemma': 'κα...",John Philoponus,In Libros De Generatione Animalium Commentaria,529,575,christian,christian_300_600,"[φημί, ζῷον, ἔχω, προβολή, ὄμμα, εἰμί, ἄνθρωπο..."
90818,lagt_tlg0057.tlg011,lagt_tlg0057.tlg011_152,152,"εἴπερ οὖν ὀρθῶς τίθενται, πολλάκις οὐκ ὀρθῇ δι...","[{'char_end': 5, 'char_start': 0, 'lemma': 'εἴ...",Galen,De anatomicis administrationibus,129,216,pagan,pagan_0_300,"[τίθημι, ὀρθός, διδασκαλία, χράω, μῦς, ἀγνοέω,..."
910866,lagt_tlg4015.tlg009,lagt_tlg4015.tlg009_13827,13827,Τὸ κινοῦν γὰρ καὶ τὸ κινούμενον τὸ μὲν τὴν ποι...,"[{'char_end': 2, 'char_start': 0, 'lemma': 'ὁ'...",John Philoponus,In Aristotelis Physicorum Libros Commentaria,529,575,christian,christian_300_600,"[κινέω, κινέω, ποιητικός, παθητικός, σχέσις, ἔχω]"
92074,lagt_tlg0057.tlg011,lagt_tlg0057.tlg011_2987,2987,καὶ πρὸς μὲν τὸ παραλυθῆναι τοῦ μεσοπλευρίου μ...,"[{'char_end': 3, 'char_start': 0, 'lemma': 'κα...",Galen,De anatomicis administrationibus,129,216,pagan,pagan_0_300,"[παραλύω, μεσοπλεύριος, μῦς, ἐνέργεια, κακός, ..."
581310,lagt_tlg1764.tlg001,lagt_tlg1764.tlg001_523,523,καταλήγει εἰς τὴν τοῦ Λέοντος μοῖραν η΄ ἔσται ...,"[{'char_end': 9, 'char_start': 0, 'lemma': 'κα...",Vettius Valens,Anthologiarum libri ix,152,162,pagan,pagan_0_300,"[καταλήγω, λέων, μοῖρα, εἰμί, ἐκλειπτικός, τόπ..."


In [61]:
# store into your personal location:
grouped.to_pickle('../data/large-data/grouped_df.pkl')
# store into our shared data directory
grouped.to_pickle('/srv/data/enemy-christ/large-data/grouped_df.pkl')

In [62]:
### later tests...
# grouped = pd.read_pickle('/srv/data/enemy-christ/large-data/grouped_df.pkl')

In [64]:
# Python
def detect_target(tokens, target):
    return any(t.get("lemma") == target for t in tokens)

target = 'ἐχθρός'

enemy_sentences_all = grouped[grouped["tokens"].apply(lambda x: detect_target(x, target))]

In [65]:
len(enemy_sentences_all)

2818